In [ ]:
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt

# Load dataset
def load_edges(filepath):
    edges = []
    max_id = 0
    with open(filepath, 'r') as file:
        for line in file:
            u, v = map(int, line.strip().split())
            edges.append((u, v))
            max_id = max(max_id, u, v)
    return edges, max_id + 1  # domain size = max ID + 1

def mean_absolute_error(true_freq, est_freq):  # Fixed function name
    return np.mean(np.abs(true_freq - est_freq))

# Direct Encoding: Perturb single integer into another in domain
def perturb_direct_encoding(value, domain_size, epsilon):
    p = np.exp(epsilon) / (np.exp(epsilon) + domain_size - 1)
    q = (1 - p) / (domain_size - 1)
    probs = [q] * domain_size
    probs[value] = p
    return np.random.choice(domain_size, p=probs)

# Estimate frequencies from perturbed data
def estimate_direct_encoding(perturbed_data, domain_size, epsilon):
    n = len(perturbed_data)
    p = np.exp(epsilon) / (np.exp(epsilon) + domain_size - 1)
    q = (1 - p) / (domain_size - 1)

    freq_counts = np.zeros(domain_size)
    for val in perturbed_data:
        freq_counts[val] += 1

    est_freq_de = (freq_counts - n * q) / (p - q)
    est_freq_de = np.clip(est_freq_de, 0, None)  # Changed est_freq to est_freq_de
    return est_freq_de / est_freq_de.sum()  # Normalize to sum=1

# Main DE execution
def run_direct_encoding(filepath, epsilon):
    edges, domain_size = load_edges(filepath)
    print(f"Loaded {len(edges)} edges, Domain size = {domain_size}")

    true_freq_sender_de = np.zeros(domain_size)
    true_freq_receiver_de = np.zeros(domain_size)
    perturbed_sender_de = []
    perturbed_receiver_de = []

    for u, v in edges:
        true_freq_sender_de[u] += 1
        true_freq_receiver_de[v] += 1
        perturbed_sender_de.append(perturb_direct_encoding(u, domain_size, epsilon))
        perturbed_receiver_de.append(perturb_direct_encoding(v, domain_size, epsilon))

    true_freq_sender_de /= true_freq_sender_de.sum()
    true_freq_receiver_de /= true_freq_receiver_de.sum()

    est_freq_sender_de = estimate_direct_encoding(perturbed_sender_de, domain_size, epsilon)
    est_freq_receiver_de = estimate_direct_encoding(perturbed_receiver_de, domain_size, epsilon)

    # Fixed variable names to use est_freq_sender_de and est_freq_receiver_de
    mae_sender = mean_absolute_error(true_freq_sender_de, est_freq_sender_de)
    mae_receiver = mean_absolute_error(true_freq_receiver_de, est_freq_receiver_de)
    mae_avg = (mae_sender + mae_receiver) / 2

    print(f"DE MAE (Sender):   {mae_sender:.6f}")
    print(f"DE MAE (Receiver): {mae_receiver:.6f}")
    print(f"DE Average MAE:    {mae_avg:.6f}")

    return {
        'true_freq_sender': true_freq_sender_de,
        'est_freq_sender': est_freq_sender_de,
        'true_freq_receiver': true_freq_receiver_de,
        'est_freq_receiver': est_freq_receiver_de,
        'mae_sender': mae_sender,
        'mae_receiver': mae_receiver,
        'mae_avg': mae_avg
    }

def perturb_ue(value, domain_size, epsilon):
    unary = np.zeros(domain_size)
    p = np.exp(epsilon / 2) / (np.exp(epsilon / 2) + 1)
    q = 1 / (np.exp(epsilon / 2) + 1)

    for i in range(domain_size):
        if i == value:
            unary[i] = 1 if np.random.rand() < p else 0
        else:
            unary[i] = 1 if np.random.rand() < q else 0
    return unary

def estimate_ue(perturbed_data, domain_size, epsilon):
    n = len(perturbed_data)
    p = np.exp(epsilon / 2) / (np.exp(epsilon / 2) + 1)
    q = 1 / (np.exp(epsilon / 2) + 1)

    aggregated = np.sum(perturbed_data, axis=0)
    est_freq_ue = (aggregated - n * q) / (p - q)
    # Changed variable name from est_freq to est_freq_ue in the following line
    est_freq_ue = np.clip(est_freq_ue, 0, None)
    # Using est_freq_ue consistently here as well
    return est_freq_ue / est_freq_ue.sum()

def run_unary_encoding(filepath, epsilon):
    edges, domain_size = load_edges(filepath)
    print(f"Loaded {len(edges)} edges, Domain size = {domain_size}")

    true_freq_sender_ue = np.zeros(domain_size)
    true_freq_receiver_ue = np.zeros(domain_size)
    perturbed_sender_ue = []
    perturbed_receiver_ue = []

    for u, v in edges:
        true_freq_sender_ue[u] += 1
        true_freq_receiver_ue[v] += 1
        perturbed_sender_ue.append(perturb_ue(u, domain_size, epsilon))
        perturbed_receiver_ue.append(perturb_ue(v, domain_size, epsilon))

    true_freq_sender_ue /= true_freq_sender_ue.sum()
    true_freq_receiver_ue /= true_freq_receiver_ue.sum()

    est_freq_sender_ue = estimate_ue(perturbed_sender_ue, domain_size, epsilon)
    est_freq_receiver_ue = estimate_ue(perturbed_receiver_ue, domain_size, epsilon)

    # Fixed variable names to use est_freq_sender_ue and est_freq_receiver_ue
    mae_sender = mean_absolute_error(true_freq_sender_ue, est_freq_sender_ue)
    mae_receiver = mean_absolute_error(true_freq_receiver_ue, est_freq_receiver_ue)
    mae_avg = (mae_sender + mae_receiver) / 2

    print(f"UE MAE (Sender):   {mae_sender:.6f}")
    print(f"UE MAE (Receiver): {mae_receiver:.6f}")
    print(f"UE Average MAE:    {mae_avg:.6f}")

    return {
        'true_freq_sender': true_freq_sender_ue,
        'est_freq_sender': est_freq_sender_ue,
        'true_freq_receiver': true_freq_receiver_ue,
        'est_freq_receiver': est_freq_receiver_ue,
        'mae_sender': mae_sender,
        'mae_receiver': mae_receiver,
        'mae_avg': mae_avg
    }

def perturb_oue(value, domain_size, epsilon):
    unary = np.zeros(domain_size)
    p = 0.5
    q = 1 / (np.exp(epsilon) + 1)

    for i in range(domain_size):
        if i == value:
            unary[i] = 1 if np.random.rand() < p else 0
        else:
            unary[i] = 1 if np.random.rand() < q else 0
    return unary

def estimate_oue(perturbed_data, domain_size, epsilon):
    n = len(perturbed_data)
    p = 0.5
    q = 1 / (np.exp(epsilon) + 1)

    aggregated = np.sum(perturbed_data, axis=0)
    est_freq_oue = (aggregated - n * q) / (p - q)
    est_freq_oue = np.clip(est_freq_oue, 0, None)  # Fixed: changed est_freq to est_freq_oue
    return est_freq_oue / est_freq_oue.sum()  # Fixed: changed est_freq to est_freq_oue

def run_optimized_unary_encoding(filepath, epsilon):
    edges, domain_size = load_edges(filepath)
    print(f"Loaded {len(edges)} edges, Domain size = {domain_size}")

    true_freq_sender_oue = np.zeros(domain_size)
    true_freq_receiver_oue = np.zeros(domain_size)
    perturbed_sender_oue = []
    perturbed_receiver_oue = []

    for u, v in edges:
        true_freq_sender_oue[u] += 1  # Fixed: changed true_freq_sender to true_freq_sender_oue
        true_freq_receiver_oue[v] += 1  # Fixed: changed true_freq_receiver to true_freq_receiver_oue
        perturbed_sender_oue.append(perturb_oue(u, domain_size, epsilon))
        perturbed_receiver_oue.append(perturb_oue(v, domain_size, epsilon))

    true_freq_sender_oue /= true_freq_sender_oue.sum()
    true_freq_receiver_oue /= true_freq_receiver_oue.sum()

    est_freq_sender_oue = estimate_oue(perturbed_sender_oue, domain_size, epsilon)
    est_freq_receiver_oue = estimate_oue(perturbed_receiver_oue, domain_size, epsilon)

    mae_sender = mean_absolute_error(true_freq_sender_oue, est_freq_sender_oue)  # Fixed: added _oue suffix
    mae_receiver = mean_absolute_error(true_freq_receiver_oue, est_freq_receiver_oue)  # Fixed: added _oue suffix
    mae_avg = (mae_sender + mae_receiver) / 2

    print(f"OUE MAE (Sender):   {mae_sender:.6f}")
    print(f"OUE MAE (Receiver): {mae_receiver:.6f}")
    print(f"OUE Average MAE:    {mae_avg:.6f}")

    return {
        'true_freq_sender': true_freq_sender_oue,
        'est_freq_sender': est_freq_sender_oue,
        'true_freq_receiver': true_freq_receiver_oue,
        'est_freq_receiver': est_freq_receiver_oue,
        'mae_sender': mae_sender,
        'mae_receiver': mae_receiver,
        'mae_avg': mae_avg
    }

def evaluate_mae_over_epsilon(filepath, epsilons):
    maes_de = []
    maes_ue = []
    maes_oue = []

    for eps in epsilons:
        print(f"\nEvaluating ε = {eps:.2f}")

        results_de = run_direct_encoding(filepath, eps)
        results_ue = run_unary_encoding(filepath, eps)
        results_oue = run_optimized_unary_encoding(filepath, eps)

        maes_de.append(results_de["mae_avg"])
        maes_ue.append(results_ue["mae_avg"])
        maes_oue.append(results_oue["mae_avg"])

    return maes_de, maes_ue, maes_oue

def plot_mae_vs_epsilon(epsilons, maes_de, maes_ue, maes_oue, save_filename=None):
    plt.figure(figsize=(10, 6))
    plt.plot(epsilons, maes_de, marker='o', label="Direct Encoding (DE)")
    plt.plot(epsilons, maes_ue, marker='s', label="Unary Encoding (UE)")
    plt.plot(epsilons, maes_oue, marker='^', label="Optimized Unary Encoding (OUE)")

    plt.xlabel("Epsilon (ε) - Privacy Level")
    plt.ylabel("Mean Absolute Error (MAE)")
    plt.title("Accuracy vs Privacy Trade-off (Lower MAE = Higher Accuracy)")
    plt.legend()
    plt.grid(True)

    if save_filename:
        plt.savefig(save_filename, dpi=300, bbox_inches='tight')
    
    plt.show()

# Define epsilon values to test
epsilons = [0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]

# Load dataset
filepath = "email-Eu-core.txt"

# Run evaluations
maes_de, maes_ue, maes_oue = evaluate_mae_over_epsilon(filepath, epsilons)

# Plot the result
plot_mae_vs_epsilon(epsilons, maes_de, maes_ue, maes_oue, save_filename="mae_vs_epsilon.png")



Evaluating ε = 0.10
Loaded 25571 edges, Domain size = 1005
DE MAE (Sender):   0.001351
DE MAE (Receiver): 0.001288
DE Average MAE:    0.001320
Loaded 25571 edges, Domain size = 1005
